# Earth–Sun distance illumination-leak diagnostic

**Companion to** `sza_leak_diagnostic.ipynb`. Besides solar-zenith geometry, the other big illumination term reflectance divides out but radiance may retain is the **Earth–Sun distance**. Top-of-atmosphere solar irradiance scales as `1/d²`, i.e. with the eccentricity factor `E0 = (d0/d)²` — about **±3.4%** across the year (perihelion early January, aphelion early July). At-sensor radiance carries that factor **unless the L1B product already normalises Earth–Sun distance**; reflectance always removes it.

**Where the data comes from:** no download needed — `E0` is a closed-form function of day-of-year (Spencer 1971), computed here from each scene's acquisition datetime (`t1_dt`/`t2_dt`, already in the aggregated tables).

This notebook (1) computes the per-scene `E0`, (2) tests whether the pair flux ratio `E0(t2)/E0(t1)` correlates with Δt, and (3) recomputes the radiance drift spectrum with the `E0` variation removed. **If EMIT L1B already normalises Earth–Sun distance the correction will barely move the spectrum — which is itself the answer.**

In [ ]:
# Repo-relative bootstrap: find the repo root (holds run_config.yaml) and import.
import sys, pathlib
_p = pathlib.Path.cwd()
while not (_p / "run_config.yaml").exists() and _p != _p.parent:
    _p = _p.parent
if str(_p) not in sys.path:
    sys.path.insert(0, str(_p))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import kendalltau, theilslopes

from emit_pics import paths
from emit_pics.run_config import load_config
from emit_pics.matched_pairs_drift_analysis import (
    load_aggregated_delta_tables, compute_drift_spectrum, DAYS_PER_YEAR)

cfg   = load_config()
ALPHA = cfg.alpha
SIZE  = cfg.patch_size

# Pool the aggregated (one-row-per-pair) delta tables across all PICS of this size.
df = load_aggregated_delta_tables(paths.DELTA_AGG_DIR, SIZE)
if df.empty:
    raise SystemExit(
        f"No aggregated delta tables for size {SIZE!r} in {paths.DELTA_AGG_DIR}.\n"
        "Run the analyze stage (scripts/run_PICS_allpatches_aggregate_pairs.py) first.")

n_pairs = df.groupby(["PICS_ID", "pair_id"]).ngroups
print(f"patch size {SIZE} km | {len(df)} rows | {df['PICS_ID'].nunique()} PICS | "
      f"{n_pairs} unique pairs | {df['band_index'].nunique()} channels")

## 1. Per-scene eccentricity factor E0 (Spencer 1971)
`E0 = (d0/d)²` is the multiplicative factor on TOA solar irradiance (and hence on at-sensor radiance if not already normalised). `E0 > 1` near perihelion (Jan), `< 1` near aphelion (Jul).

In [ ]:
def spencer_E0(dt_series):
    # Eccentricity correction factor E0 = (d0/d)^2 from acquisition datetimes (Spencer 1971).
    doy = pd.to_datetime(dt_series).dt.dayofyear.to_numpy().astype(float)
    g = 2.0 * np.pi * (doy - 1.0) / 365.0
    return (1.000110 + 0.034221 * np.cos(g) + 0.001280 * np.sin(g)
            + 0.000719 * np.cos(2 * g) + 0.000077 * np.sin(2 * g))

df = df.copy()
df["E0_t1"] = spencer_E0(df["t1_dt"])
df["E0_t2"] = spencer_E0(df["t2_dt"])

pairs = df.drop_duplicates(subset=["PICS_ID", "pair_id"]).copy()
pairs["flux_ratio_pct"] = 100.0 * (pairs["E0_t2"] / pairs["E0_t1"] - 1.0)
print(f"{len(pairs)} unique pairs")
print(f"E0 range across scenes : [{df['E0_t1'].min():.4f}, {df['E0_t1'].max():.4f}]  "
      f"(annual span ~ {100*(df['E0_t1'].max()/df['E0_t1'].min()-1):.1f}%)")
print(f"flux ratio E0(t2)/E0(t1) : mean {pairs['flux_ratio_pct'].mean():+.3f}%   "
      f"range [{pairs['flux_ratio_pct'].min():+.2f}, {pairs['flux_ratio_pct'].max():+.2f}]%")

## 2. Does the Earth–Sun flux ratio correlate with Δt?
A Δt-correlated flux ratio would map into an apparent radiance drift. Because the seasonal cycle is annual, correlation with Δt appears when the pair sampling is not seasonally balanced across baselines.

In [ ]:
m = pairs["flux_ratio_pct"].notna() & pairs["dt_days"].notna()
x = pairs.loc[m, "dt_days"].to_numpy()
y = pairs.loc[m, "flux_ratio_pct"].to_numpy()
tau, p = kendalltau(x, y)
slope, intc, lo, hi = theilslopes(y, x)
print(f"flux ratio (%) vs Δt :  Theil–Sen slope {slope * DAYS_PER_YEAR:+.4f} %/yr   "
      f"Kendall τ = {tau:+.3f}   p = {p:.2g}")

fig, ax = plt.subplots(1, 2, figsize=(13, 4.5))
ax[0].scatter(x, y, s=12, alpha=0.5)
xr = np.array([x.min(), x.max()])
ax[0].plot(xr, intc + slope * xr, "k-", lw=1.5)
ax[0].axhline(0.0, color="0.6", lw=0.8)
ax[0].set_xlabel("Δt (days)")
ax[0].set_ylabel("100·(E0(t2)/E0(t1) − 1)  [%]")
ax[0].set_title(f"Earth–Sun flux ratio vs Δt   (τ={tau:+.2f}, p={p:.2g})")
ax[1].hist(y, bins=40, color="tab:purple", alpha=0.85)
ax[1].axvline(0.0, color="k", lw=1.0)
ax[1].axvline(float(np.mean(y)), color="tab:red", lw=1.5, ls="--",
             label=f"mean {np.mean(y):+.2f}%")
ax[1].set_xlabel("flux ratio [%]")
ax[1].set_ylabel("pairs")
ax[1].set_title("distribution of Earth–Sun flux ratio")
ax[1].legend()
plt.tight_layout()
plt.show()

## 3. Earth–Sun (E0) correction
Divide the flux ratio `E0(t2)/E0(t1)` out of each pair's radiance % change and refit. A large change ⇒ EMIT L1B is **not** Earth–Sun normalised and this is a real leak; a negligible change ⇒ it already is (and the SZA notebook is the place to look).

In [ ]:
def remove_illumination(delta_pct, ratio):
    # Undo a multiplicative illumination factor  ratio = illum(t2) / illum(t1)
    # from a percent change. Measured (1 + dR/100) = ratio * (true ratio), so the
    # illumination-free percent change is 100 * ((1 + dR/100) / ratio - 1).
    delta_pct = np.asarray(delta_pct, dtype=float)
    ratio     = np.asarray(ratio, dtype=float)
    return 100.0 * ((1.0 + delta_pct / 100.0) / ratio - 1.0)


def median_drift(spec):
    return float(np.nanmedian(spec["slope_pct_yr"]))


def plot_spectra(specs_labels_colors, title):
    # specs_labels_colors: list of (spectrum_df, label, color)
    fig, ax = plt.subplots(figsize=(12, 5))
    for spec, lab, c in specs_labels_colors:
        s = spec.dropna(subset=["slope_pct_yr"]).sort_values("wavelength")
        ax.plot(s["wavelength"], s["slope_pct_yr"], ".", ms=5, color=c, label=lab)
    ax.axhline(0.0, color="k", lw=0.8)
    ax.set_xlabel("wavelength (nm)")
    ax.set_ylabel("drift (%/yr)")
    ax.set_title(title)
    ax.legend()
    plt.tight_layout()
    plt.show()

In [ ]:
f_es = (df["E0_t2"] / df["E0_t1"]).to_numpy()

df_corr = df.copy()
df_corr["delta_rad_pct_median"] = remove_illumination(df["delta_rad_pct_median"], f_es)

spec_rad      = compute_drift_spectrum(df,      "rad", alpha=ALPHA)
spec_rad_corr = compute_drift_spectrum(df_corr, "rad", alpha=ALPHA)
spec_rfl      = compute_drift_spectrum(df,      "rfl", alpha=ALPHA)

print(f"median radiance drift  original ......... {median_drift(spec_rad):+.3f} %/yr")
print(f"median radiance drift  E0-corrected ..... {median_drift(spec_rad_corr):+.3f} %/yr")
print(f"median reflectance drift (reference) .... {median_drift(spec_rfl):+.3f} %/yr")

In [ ]:
plot_spectra(
    [(spec_rad,      "radiance — original",       "tab:red"),
     (spec_rad_corr, "radiance — E0-corrected",   "tab:purple"),
     (spec_rfl,      "reflectance (reference)",   "tab:green")],
    "Radiance drift spectrum: raw vs Earth–Sun (E0) corrected")

## Bonus: combined cos(SZA)·E0 illumination correction
The full first-order illumination factor is the **product** of the two ratios. This cell removes both at once — the closest matched-pairs analog to what the BRDF model does explicitly.

In [ ]:
f_both = ((np.cos(np.radians(df["SZA_t2"])) / np.cos(np.radians(df["SZA_t1"])))
          * (df["E0_t2"] / df["E0_t1"])).to_numpy()

df_both = df.copy()
df_both["delta_rad_pct_median"] = remove_illumination(df["delta_rad_pct_median"], f_both)
spec_rad_both = compute_drift_spectrum(df_both, "rad", alpha=ALPHA)

print(f"median radiance drift  cos(SZA)·E0-corrected .. {median_drift(spec_rad_both):+.3f} %/yr")
plot_spectra(
    [(spec_rad,      "radiance — original",              "tab:red"),
     (spec_rad_both, "radiance — cos(SZA)·E0-corrected", "tab:orange"),
     (spec_rfl,      "reflectance (reference)",          "tab:green")],
    "Radiance drift spectrum: raw vs full illumination-corrected")

## How to read this
- **E0 correction barely moves the radiance spectrum** → EMIT L1B already normalises Earth–Sun distance; the residual positive bias is not from `1/d²`. Check the SZA notebook.
- **E0 correction removes the positive bias** → the leak is the seasonal solar-flux term, exposed because the all-patches pool samples baselines with a seasonal imbalance.
- The **combined** panel is the fullest first-order illumination correction available from matched pairs; if radiance still drifts after it, that residual is the better candidate for genuine sensor behaviour — best confirmed with the pooled BRDF fit.